# 04-burst-check: does the change rate survive artifact handling?

The final run found changes in 27.9% of scans. This repeats it three ways:

- **raw:** the same as 03-final, as a check that it reproduces.
- **clipped:** each ICN limited to 3 robust SDs.
- **censored:** volumes inside a burst are removed.

If the rate holds, the result is not driven by bursts.

**To run:** attach the data, then Save & Run All (about 20-30 minutes on CPU).

In [1]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as sch
from joblib import Parallel, delayed
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap

SEED = 20261031
INPUT_ROOT = Path(os.environ.get("DFCPD_INPUT", "/kaggle/input"))
ALPHA, L, COPIES, MIN_SEGMENT, GUARD, ESS = 0.05, 5, 99, 120, 5, 0.71
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#6f6e69"
DIV = LinearSegmentedColormap.from_list("div", ["#184f95", "#6da7ec", "#f0efec", "#ec8a8a", "#b52f2f"])
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "font.size": 9})

## Load scans

Exact copies are dropped. Each scan is z-scored, which removes the mixed scaling.

In [2]:
def zscore(x):
    return (x - x.mean(0)) / x.std(0, ddof=1)


SCANS, GROUP, seen = {}, {}, set()
for path in sorted(INPUT_ROOT.rglob("icn_tc.npy")):
    x = zscore(np.load(path).astype(np.float64))
    key = hashlib.sha1(x.round(4).tobytes()).hexdigest()
    if key not in seen:
        seen.add(key)
        SCANS[path.parent.name] = x
        GROUP[path.parent.name] = path.parent.parent.name
assert SCANS, "attach the data"
print(len(SCANS), "scans")

781 scans


## Detector

The statistic is Li & Li's whole-scan covariance change statistic. A scan's
p-value compares it with no-change copies that keep the scan's spectrum and its
slow variance trend. A significant scan is split at the best point and each side
is tested again, so one scan can have several change points.

In [3]:
def profile(x):
    n = len(x)
    g = (x @ x.T) ** 2
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    g[np.abs(i - j) < L + 1] = 0
    c = np.zeros((n + 1, n + 1))
    c[1:, 1:] = g.cumsum(0).cumsum(1)
    t = np.arange(L + 2, n - L - 1)
    left, cross = c[t, t], c[t, n] - c[t, t]
    right = c[n, n] - c[t, n] - c[n, t] + left
    value = (left / ((t - L) * (t - L - 1)) + right / ((n - t - L) * (n - t - L - 1))
             - 2 * cross / (t * (n - t) - L * (L + 1) / 2))
    return t, (t - L) * (n - t - L) * value


def statistic(x):
    return profile(x)[1].sum() / len(x) ** 2


def scale_trend(x, bandwidth=60.0):
    energy = (x ** 2).mean(1)
    gap = np.arange(len(x))[None, :] - np.arange(len(x))[:, None]
    k = np.exp(-.5 * (gap / bandwidth) ** 2)
    s0, s1, s2 = k.sum(1), (k * gap).sum(1), (k * gap ** 2).sum(1)
    smooth = np.clip((s2 * (k @ energy) - s1 * ((k * gap) @ energy)) / (s0 * s2 - s1 ** 2), .05 * energy.mean(), None)
    return np.sqrt(smooth / smooth.mean())


def no_change_copy(x, rng):
    s = scale_trend(x)
    flat = zscore(x / s[:, None])
    spectrum = np.fft.rfft(flat, axis=0)
    phases = np.exp(2j * np.pi * rng.random(spectrum.shape[0]))
    phases[0] = 1
    if len(x) % 2 == 0:
        phases[-1] = rng.choice((-1.0, 1.0))
    return zscore(np.fft.irfft(spectrum * phases[:, None], n=len(x), axis=0) * s[:, None])


def p_value(x, rng):
    observed = statistic(x)
    return (1 + sum(statistic(no_change_copy(x, rng)) >= observed for _ in range(COPIES))) / (COPIES + 1)


def detect(x, rng):
    found, first = [], []

    def recurse(start, stop):
        if stop - start < MIN_SEGMENT:
            return
        segment = zscore(x[start:stop])
        p = p_value(segment, rng)
        first.append(p)
        if p > ALPHA:
            return
        t, values = profile(segment)
        tau = start + int(np.clip(t[np.argmax(values)], MIN_SEGMENT // 2, len(segment) - MIN_SEGMENT // 2))
        found.append(tau)
        recurse(start, tau)
        recurse(tau, stop)

    recurse(0, len(x))
    return sorted(found), first[0] if first else np.nan


def run_scan(sid, x, seed):
    rng = np.random.default_rng(seed)
    points, p = detect(x, rng)
    null_points, null_p = detect(no_change_copy(x, rng), rng)
    return {"scan": sid, "group": GROUP[sid], "T": len(x), "change_points": points, "p": p,
            "null_change_points": null_points, "null_p": null_p}

## Run all three versions

In [4]:
BURST_LEVEL, EDGE = 2.91, 60


def clipped(x):
    median = np.median(x, 0)
    mad = np.median(np.abs(x - median), 0) * 1.4826
    return zscore(np.clip(x, median - 3 * mad, median + 3 * mad))


def burst_mask(x):
    smooth = pd.Series((x ** 2).mean(1)).rolling(5, center=True, min_periods=1).mean().to_numpy()
    return np.convolve(smooth > BURST_LEVEL, np.ones(5), "same") > 0


def censored(x):
    return zscore(x[~burst_mask(x)])


VERSIONS = {"raw": lambda x: x, "clipped": clipped, "censored": censored}


def run_version(sid, x, name, seed):
    rng = np.random.default_rng(seed)
    y = VERSIONS[name](x)
    points, _ = detect(y, rng)
    null_points, _ = detect(no_change_copy(y, rng), rng)
    return {"scan": sid, "version": name, "T": len(y), "K": len(points), "null_K": len(null_points),
            "has_burst": bool(burst_mask(x).any()), "burst_volumes": int(burst_mask(x).sum()),
            "at_limit": sum(min(t, len(y) - t) <= EDGE for t in points)}


jobs = [(sid, x, name, SEED + 1000 * v + k) for v, name in enumerate(VERSIONS) for k, (sid, x) in enumerate(SCANS.items())]
rows = Parallel(n_jobs=os.cpu_count() or 1, verbose=5)(delayed(run_version)(*job) for job in jobs)
results = pd.DataFrame(rows)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    5.6s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:   23.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   54.8s
[Parallel(n_jobs=4)]: Done 280 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:  2.5min
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed:  3.5min
[Parallel(n_jobs=4)]: Done 874 tasks      | elapsed:  4.7min
[Parallel(n_jobs=4)]: Done 1144 tasks      | elapsed:  6.2min
[Parallel(n_jobs=4)]: Done 1450 tasks      | elapsed:  7.8min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed:  9.5min
[Parallel(n_jobs=4)]: Done 2170 tasks      | elapsed: 11.5min
[Parallel(n_jobs=4)]: Done 2343 out of 2343 | elapsed: 12.3min finished


## Results

In [5]:
def rate(frame):
    return pd.Series({"scans": len(frame), "share_with_change": (frame["K"] > 0).mean(),
                      "copies_false_alarm": (frame["null_K"] > 0).mean(), "changes": frame["K"].sum(),
                      "share_changes_at_edge_limit": frame["at_limit"].sum() / max(frame["K"].sum(), 1)})


print(results.groupby("version", sort=False).apply(rate, include_groups=False).round(3).to_string())
print("\nBy whether the scan has a burst:")
print(results.groupby(["version", "has_burst"], sort=False).apply(rate, include_groups=False).round(3).to_string())
print("\nMedian volumes removed in burst scans:", results.loc[results["has_burst"], "burst_volumes"].median())

          scans  share_with_change  copies_false_alarm  changes  share_changes_at_edge_limit
version                                                                                     
raw       781.0              0.279               0.029    259.0                        0.502
clipped   781.0              0.307               0.017    299.0                        0.495
censored  781.0              0.287               0.027    269.0                        0.506

By whether the scan has a burst:
                    scans  share_with_change  copies_false_alarm  changes  share_changes_at_edge_limit
version  has_burst                                                                                    
raw      False      459.0              0.277               0.026    147.0                        0.497
         True       322.0              0.283               0.034    112.0                        0.509
clipped  False      459.0              0.296               0.015    164.0                